### Week2 - Day 1 - Other Models Client API and chat conversations

In [ ]:
import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
from google import genai
from IPython.display import Markdown, display

In [ ]:
load_dotenv(override=True)
google_api_key = os.getenv('GOOGLE_API_KEY')

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
openai = OpenAI(base_url=GEMINI_BASE_URL,api_key=google_api_key)
AGENT_MODEL = "gemini-3.1-pro"
SUBAGENT_MODEL = "gemini-3.1-flash-lite"

In [ ]:
messages = [{
    "role": "user", "content": "What is the main difference between Inference and Training in LLM?"
}]

# Call OpenAI Client API
response = openai.chat.completions.create(
    model="gemini-3.1-flash-lite",
    messages=messages
)

# Print the result
display(Markdown(response.choices[0].message.content))

In [ ]:
# Provide the same prompt to Google's Client API

client = genai.Client()

response = client.models.generate_content(
    model="gemini-3.1-flash-lite", contents=messages[0]['content']
)
display(Markdown(response.text))

In [ ]:
# Provide the same prompt using Langchain's Google API
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite")
new_response = llm.invoke(messages)

display(Markdown(new_response.text))

In [ ]:
# from langchain_community.callbacks import 
print(f"Input tokens: {new_response.usage_metadata['input_tokens']}")
print(f"Output tokens: {new_response.usage_metadata['output_tokens']}")
print(f"Total tokens: {new_response.usage_metadata['total_tokens']}")

In [ ]:
# Create a conversation between 2 Gemini Models, One as Agent and another as Subagent.

gemini_model_agent = "gemini-3.7-flash"
gemini_model_subagent = "gemini-3.1-flash-lite"

gemini_agent_sp = "You are a chatbot agent who controls another subagent of lower capability model 3.1-flash-lite; \
you need to provide specific instructions of how to plan the complex systems as the model struggles to grab the idea of the query because it lacks the training.\
Use inference technique to guide the model and train it as a better way to communicate with teh user."

gemini_subagent_sp = "You are a newly small trained subagent, whose task is to learn from the agent and provide answer for complex queries to the user. You try to learn and grab the idea of \
what the agent says to you. You are free to ask the questions for your clarifications and you must learn and grasp everything and should provide tobust answer to the user's query"

gemini_agent_messages = ["Hi there, How can you train you!"]
gemini_subagent_messages = ["Hi, I am the sub agent 3.1 flash lite"]

In [ ]:
def call_gemini_flash():
    messages = [{"role": "system", "content": gemini_agent_sp}]
    for agent, subagent in zip(gemini_agent_messages, gemini_subagent_messages):
        messages.append({"role": "assistant", "content": agent})
        messages.append({"role": "user", "content": subagent})
    response = openai.chat.completions.create(model=gemini_model_agent, messages=messages)
    return response.choices[0].message.content

In [ ]:
call_gemini_flash()

In [ ]:
def call_gemini_flash_lite():
    messages = [{"role": "system", "content": gemini_subagent_sp}]
    for agent, subagent in zip(gemini_agent_messages, gemini_subagent_messages):
        messages.append({"role": "user", "content": agent})
        messages.append({"role": "assistant", "content": subagent})
    messages.append({"role": "user", "content": gemini_agent_messages[-1]})
    response = openai.chat.completions.create(model=gemini_model_subagent, messages=messages)
    return response.choices[0].message.content

In [ ]:
call_gemini_flash_lite()

In [ ]:
for i in range(5):
    agent_next = call_gemini_flash()
    display(Markdown(f"### Agent:\n{agent_next}\n"))
    gemini_agent_messages.append(agent_next)
    
    subagent_next = call_gemini_flash_lite()
    display(Markdown(f"### Sub Agent:\n{subagent_next}\n"))
    gemini_subagent_messages.append(subagent_next)

### Week 2 - Day2 - Gradio and stream using gradio

In [ ]:
response = client.models.generate_content(
    model="gemini-3.1-flash-lite", contents="What is today's date?"
)
display(Markdown(response.text))

In [ ]:
messages = [{
    "role": "user", "content": "What is today's date?"
}]

response = openai.chat.completions.create(
    model="gemini-3.1-flash-lite",
    messages=messages
)

display(Markdown(response.choices[0].message.content))

In [ ]:
import gradio as gr

In [ ]:
# Gradio UI usage - Example
system_message = "You are a helpful assistant"
def message_gemini(prompt):
    messages = [{"role": "system", "content": system_message}, {"role": "user", "content": prompt}]
    response = openai.chat.completions.create(model="gemini-3.5-flash-lite", messages=messages)
    return response.choices[0].message.content

In [ ]:
force_dark_mode = """
function refresh() {
    const url = new URL(window.location);
    if (url.searchParams.get('__theme') !== 'dark') {
        url.searchParams.set('__theme', 'dark');
        window.location.href = url.href;
    }
}
"""

In [ ]:
gr.Interface(fn=message_gemini, inputs="textbox", outputs="textbox", flagging_mode="never", js=force_dark_mode).launch()

In [ ]:
# Stream the Gemini Response in GradioUI
system_message = "You are a helpful assistant that responds in markdown without code blocks"

def stream_response_gemini(prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
      ]
    stream = openai.chat.completions.create(
        model='gemini-3.5-flash-lite',
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [ ]:
message_input = gr.Textbox(label="Your Query:", info="Enter your query for Gemini Model API", lines=7)
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_response_gemini,
    title="Gemini API", 
    inputs=[message_input], 
    outputs=[message_output], 
    examples=[
        "Explain the Transformer architecture to a layman",
        "Explain the Transformer architecture to an aspiring AI engineer",
        ], 
    flagging_mode="never"
    )
view.launch()

### Week2 - Day3 - Build a straming chatbot using Gradio UI

In [ ]:
# Define a chat function with message and history as the param. Gradio uses history as similar to openai in Gradio ChatInterface
def chat(message, history):
    history = [{"role": h['role'], "content":h['content']} for h in history]
    message = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    stream = openai.chat.completions.create(model="gemini-3.5-flash-lite", messages=message, stream=True)
    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [ ]:
gr.ChatInterface(fn=chat).launch()

### Week 2 - Day 4 - Tool Calling

In [ ]:
import json

In [ ]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [ ]:
# Create a function that can be used for tool calling for FlightAI

ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(destination_city):
    print(f"Tool called for city {destination_city}")
    price = ticket_prices.get(destination_city.lower(), "Unknown ticket price")
    return f"The price of a ticket to {destination_city} is {price}"

In [ ]:
get_ticket_price("London")

In [ ]:
# Dictionary structure that's required to describe the tool function defined earlier

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

In [ ]:
tools = [{"type": "function", "function": price_function}]

# We have to write that function handle_tool_calls to handle multiple calls:

def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [ ]:
# Update the chat function accordinlgy to handle multiple tool calls

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gemini-3.5-flash-lite", messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model="gemini-3.5-flash-lite", messages=messages)
    
    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(fn=chat).launch()

In [ ]:
# Instead of using the standard data dictonary we can use SQlite3 for storing the price db
import sqlite3

DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute("CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)")
    conn.commit()

In [ ]:
# function for inserting the data into table

def set_ticket_price(city, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()

In [ ]:
# Insering the data to the table

ticket_prices = {"london":799, "paris": 899, "tokyo": 1420, "sydney": 2999}
for city, price in ticket_prices.items():
    set_ticket_price(city, price)

In [ ]:
# Function to fetch the result with city and price
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [ ]:
get_ticket_price("Tokyo")

In [ ]:
# Re-run the chatinterface function for gradio as the get_ticket_price function is updated using DB
gr.ChatInterface(fn=chat).launch(server_name="0.0.0.0") # Server Name - To use it in our other network devices

### Week 2 - Day 5 - Multi Models

In [ ]:
# Some imports for handling images
import base64
from io import BytesIO
from PIL import Image
import gradio as gr

In [ ]:
# ----- Not Working ------ #
# def artist(city):
#     image_response = openai.images.generate(
#             model="gemini-3.1-flash-lite-image",
#             prompt=f"""Monochromatic black-and-white expressive sketch of {city} skyline, 
#             loose digital painting style with visible brushstrokes and varying ink densities, 
#             sharp geometric shapes for buildings, prominent central landmark as focal point, 
#             street-level foreground with loose gestural strokes in the lower third, 
#             bright washed-out daytime sky, dynamic and slightly abstract feel, strictly grayscale palette only, 
#             no color, wide landscape format.""",
#             # size="1024x1024",
#             response_format='b64_json',
#             n=1,
#         )
#     image_base64 = image_response.data[0].b64_json
#     image_data = base64.b64decode(image_base64)
#     return Image.open(BytesIO(image_data))

In [ ]:
# ----- Not Working ------ #
# image = artist("New York City")
# display(image)

In [ ]:
# ----- Not Working ------ #
# from google import genai
# import base64

# client = genai.Client()

# interaction = client.interactions.create(
#     model="gemini-3.1-flash-lite-image",
#     input="Create a vibrant infographic that explains photosynthesis as if it were a recipe for a plant's favorite food. Show the \"ingredients\" (sunlight, water, CO2) and the \"finished dish\" (sugar/energy). The style should be like a page from a colorful kids' cookbook, suitable for a 4th grader.",
#     generation_config={"thinking_level": "minimal"},
#     tools=[{"type": "google_search"}],
# )

# with open("photosynthesis.png", "wb") as f:
#     f.write(base64.b64decode(interaction.output_image.data))

In [ ]:
client = genai.Client()

def talker(message):
    interaction = client.interactions.create(
        model="gemini-3.8-flash-tts",
        input=[{
            "type": "user_input",
            "content": [{
                "type": "text",
                "text": message,
            }],
        }],
        response_format={"type": "audio"},
        generation_config={
            "speech_config": [
                {"voice": "Charon"},
            ]
        },
    )
    return base64.b64decode(interaction.output_audio.data)

In [ ]:
def handle_tool_calls_and_return_cities(message):
    responses = []
    cities = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            cities.append(city)
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses, cities

In [ ]:
def chat(history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history
    response = openai.chat.completions.create(model="gemini-3.1-flash-lite", messages=messages, tools=tools)
    cities = []
    image = None

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses, cities = handle_tool_calls_and_return_cities(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model="gemini-3.1-flash-lite", messages=messages, tools=tools)

    reply = response.choices[0].message.content
    history += [{"role":"assistant", "content":reply}]

    voice = talker(reply)

    # if cities:
    #     image = artist(cities[0])
    
    # return history, voice, image
    return history, voice


In [ ]:
# Callbacks (along with the chat() function above)

def put_message_in_chatbot(message, history):
        return "", history + [{"role":"user", "content":message}]

# UI definition

with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500)
        # image_output = gr.Image(height=500, interactive=False)
    with gr.Row():
        audio_output = gr.Audio(autoplay=True)
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant:")

# Hooking up events to callbacks

    message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
        chat, inputs=chatbot, outputs=[chatbot, audio_output]
    )

ui.launch(inbrowser=True)